# Shinobu Colab GPU 声带 Worker v2（IndexTTS-2.5 · 自动报到热插拔）

支持 **Google Colab T4 / L4 / A100** 自动精度适配（L4/A100 自动 `bf16`，T4 自动 `fp16`），启动后通过 **Cloudflare 隧道** 或 **Tailscale userspace** 暴露 `/v1/audio/speech`（原生支持 `ogg_opus` / `mp3` / `wav`），并**自动向云端 `/api/workers/register` 与 `/api/workers/heartbeat` 报到**，彻底告别手动拷贝隧道链接！

### 使用步骤（首次约 3~5 分钟，后续走 Drive 缓存约 1 分钟）
1. 顶部菜单 **代码执行程序 → 更改运行时类型 → 选择 T4 或 L4 GPU**。
2. 推荐在 Colab 左侧钥匙图标 **Secrets** 中配置 `SHINOBU_CLOUD_URL`、`SHINOBU_CLOUD_TOKEN`（若用 Tailscale 再配 `TS_AUTHKEY`），也可直接在 Cell 4 表单填写。
3. 直接点 **全部运行**：Cell 3 会优先复用 Drive 的 `ref.wav`，Cell 4 启动后自动注册到云端 `WorkerRegistry`，云端 `TtsRouter` 秒切满血声线；关闭 Notebook 时自动 `unregister`，无感切回 GCP `Chirp 3: HD`。


In [ ]:
#@title 1. 检查 T4/L4 GPU、可选挂载 Google Drive 并安装运行环境
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/shinobu_tts"  #@param {type:"string"}
TRANSPORT_MODE = "cloudflare"  #@param ["cloudflare", "tailscale"]

import subprocess
from pathlib import Path

!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
    print(f"[OK] Google Drive cache dir: {DRIVE_ROOT}")

!pip install -q -U uv "huggingface-hub[cli]"

if not Path("/content/index-tts").exists():
    !git clone --depth 1 https://github.com/index-tts/index-tts.git /content/index-tts

%cd /content/index-tts
!uv sync --all-extras

if TRANSPORT_MODE == "cloudflare":
    if not Path("/usr/local/bin/cloudflared").exists():
        !curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
        !chmod +x /usr/local/bin/cloudflared
    print("[OK] cloudflared ready!")
else:
    if not Path("/usr/bin/tailscale").exists() and not Path("/usr/local/bin/tailscale").exists():
        !curl -fsSL https://tailscale.com/install.sh | sh
    print("[OK] tailscale binary ready (will run in --tun=userspace-networking mode)!")

print("[OK] Environment ready!")


In [ ]:
#@title 2. 下载或复用 IndexTTS-2.5 模型权重（优先走 Drive 缓存）
import shutil
from pathlib import Path

ckpt_dir = Path(DRIVE_ROOT) / "checkpoints_v25" if USE_DRIVE else Path("/content/index-tts/checkpoints")
ckpt_dir.mkdir(parents=True, exist_ok=True)

if not (ckpt_dir / "config.yaml").is_file():
    print(f"[Downloading] IndexTeam/IndexTTS-2.5 -> {ckpt_dir} ...")
    !hf download IndexTeam/IndexTTS-2.5 --local-dir="{ckpt_dir}"
else:
    print(f"[Cache Hit] Reusing weights from {ckpt_dir}")

local_ckpt = Path("/content/index-tts/checkpoints")
if str(ckpt_dir.resolve()) != str(local_ckpt.resolve()):
    if local_ckpt.is_symlink() or local_ckpt.is_file():
        local_ckpt.unlink()
    elif local_ckpt.is_dir():
        shutil.rmtree(local_ckpt)
    local_ckpt.symlink_to(ckpt_dir)

print("[OK] Checkpoints ready:", sorted(p.name for p in local_ckpt.iterdir())[:10])


In [ ]:
#@title 3. 准备参考干音（5~15 秒干净人声，优先复用 Drive）
import subprocess
from pathlib import Path

ref_wav = Path(DRIVE_ROOT) / "ref.wav" if USE_DRIVE else Path("/content/ref.wav")
FORCE_REUPLOAD = False  #@param {type:"boolean"}

if ref_wav.is_file() and not FORCE_REUPLOAD:
    print(f"[OK] Using cached reference voice: {ref_wav} ({ref_wav.stat().st_size} bytes)")
else:
    from google.colab import files
    print("请上传一段 5~15 秒无背景音乐的干净人声参考音频...")
    uploaded = files.upload()
    assert uploaded, "未选择参考音频文件！"
    src_name = next(iter(uploaded.keys()))
    src_path = Path("/content") / src_name
    if not src_path.exists():
        src_path = Path.cwd() / src_name
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(src_path), "-ac", "1", "-ar", "24000", str(ref_wav)],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    print(f"[OK] Normalized and saved reference voice -> {ref_wav}")


In [ ]:
#@title 4. 启动 T4/L4 OpenAI /v1/audio/speech 服务 + 自动云端报到与心跳守护
CLOUD_BASE_URL = ""  #@param {type:"string"}
CLOUD_ADMIN_TOKEN = ""  #@param {type:"string"}
TS_AUTHKEY = ""  #@param {type:"string"}
TS_HOSTNAME = "shinobu-colab-tts"  #@param {type:"string"}
WORKER_ID = "colab-gpu-tts"  #@param {type:"string"}
GPU_PRECISION = "auto"  #@param ["auto", "bf16", "fp16"]
HEARTBEAT_INTERVAL_SEC = 20  #@param {type:"integer"}
WORKER_TTL_SEC = 60  #@param {type:"integer"}

import json
import os
import re
import secrets
import subprocess
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

def _get_secret(name: str, fallback: str = "") -> str:
    if fallback.strip():
        return fallback.strip()
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return str(val).strip()
    except Exception:
        pass
    return os.environ.get(name, "").strip()

def _detect_gpu() -> tuple[str, str]:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
            text=True,
        ).strip().splitlines()[0]
        parts = [p.strip() for p in out.split(",")]
        gpu_name = parts[0]
        cap = float(parts[1]) if len(parts) > 1 else 7.5
    except Exception:
        gpu_name, cap = "unknown-gpu", 7.5
    if GPU_PRECISION in ("bf16", "fp16"):
        return gpu_name, GPU_PRECISION
    return gpu_name, ("bf16" if cap >= 8.0 else "fp16")

CLOUD_URL = _get_secret("SHINOBU_CLOUD_URL", CLOUD_BASE_URL).rstrip("/")
CLOUD_TOKEN = _get_secret("SHINOBU_CLOUD_TOKEN", CLOUD_ADMIN_TOKEN)
TAILSCALE_KEY = _get_secret("TS_AUTHKEY", TS_AUTHKEY)
GPU_NAME, PRECISION = _detect_gpu()
API_KEY = "shinobu-gpu-" + secrets.token_urlsafe(16)
REF_WAV = str(Path(DRIVE_ROOT) / "ref.wav" if USE_DRIVE else Path("/content/ref.wav"))
PORT = 8080
assert Path(REF_WAV).is_file(), f"找不到参考干音 {REF_WAV}，请先运行 Cell 3！"
print(f"[GPU] {GPU_NAME} | precision={PRECISION} | transport={TRANSPORT_MODE}")

server_code = r'''
import json, os, re, subprocess, sys, tempfile, threading, time
from http import HTTPStatus
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from indextts.infer_v2_5 import IndexTTS2

API_KEY = os.environ.get("SHINOBU_TTS_API_KEY", "").strip()
REF_WAV = os.environ["SHINOBU_REF_WAV"]
PORT = int(os.environ.get("SHINOBU_TTS_PORT", "8080"))
BIND_HOST = os.environ.get("SHINOBU_TTS_BIND", "127.0.0.1")
PRECISION = os.environ.get("SHINOBU_GPU_PRECISION", "bf16").strip().lower()
GPU_NAME = os.environ.get("SHINOBU_GPU_NAME", "cuda")

_JA_RE = re.compile(r"[\u3040-\u30ff]")
_ZH_RE = re.compile(r"[\u4e00-\u9fff]")

def detect_lang(text: str) -> str:
    if _JA_RE.search(text):
        return "JA"
    if _ZH_RE.search(text):
        return "ZH"
    return "EN"

def transcode_audio(wav_path: str, req_fmt: str) -> tuple[bytes, str]:
    fmt = str(req_fmt or "wav").strip().lower()
    if fmt in ("opus", "ogg_opus", "ogg"):
        out_path = wav_path + ".ogg"
        try:
            subprocess.run(
                ["ffmpeg", "-y", "-i", wav_path, "-c:a", "libopus", "-b:a", "48k", "-vbr", "on", out_path],
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            return Path(out_path).read_bytes(), "audio/ogg"
        finally:
            if os.path.exists(out_path):
                os.remove(out_path)
    if fmt in ("mp3", "mpeg"):
        out_path = wav_path + ".mp3"
        try:
            subprocess.run(
                ["ffmpeg", "-y", "-i", wav_path, "-c:a", "libmp3lame", "-q:a", "2", out_path],
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            return Path(out_path).read_bytes(), "audio/mpeg"
        finally:
            if os.path.exists(out_path):
                os.remove(out_path)
    return Path(wav_path).read_bytes(), "audio/wav"

use_bf16 = PRECISION == "bf16"
tts_kwargs = {
    "cfg_path": "checkpoints/config.yaml",
    "model_dir": "checkpoints",
    "use_bf16": use_bf16,
    "use_qwen_emo": False,
}
if not use_bf16:
    tts_kwargs["use_fp16"] = True

print(f"[IndexTTS-2.5] Loading on {GPU_NAME} (precision={PRECISION})...", flush=True)
try:
    tts = IndexTTS2(**tts_kwargs)
except TypeError:
    tts_kwargs.pop("use_fp16", None)
    tts = IndexTTS2(**tts_kwargs)

infer_lock = threading.Lock()
stats = {"requests": 0, "started_at": time.time()}
print("[IndexTTS-2.5] Model loaded! Warming up...", flush=True)
with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tf:
    warm_path = tf.name
try:
    tts.infer(spk_audio_prompt=REF_WAV, text="吾乃怪异之王。", output_path=warm_path, lang="ZH")
    print("[IndexTTS-2.5] Warmup done!", flush=True)
finally:
    if os.path.exists(warm_path):
        os.remove(warm_path)

class Handler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        sys.stdout.write(f"[GPU-TTS] {fmt % args}\n")
        sys.stdout.flush()

    def _cors(self):
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Authorization, Content-Type")

    def _json(self, status, obj):
        raw = json.dumps(obj, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self._cors()
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(raw)))
        self.end_headers()
        self.wfile.write(raw)

    def _check_auth(self) -> bool:
        if not API_KEY:
            return True
        return self.headers.get("Authorization", "") == f"Bearer {API_KEY}"

    def do_OPTIONS(self):
        self.send_response(HTTPStatus.NO_CONTENT)
        self._cors()
        self.send_header("Content-Length", "0")
        self.end_headers()

    def do_GET(self):
        path = self.path.split("?", 1)[0].rstrip("/") or "/"
        if path in ("/", "/health"):
            self._json(200, {
                "status": "ok",
                "model": "IndexTTS-2.5",
                "gpu": GPU_NAME,
                "precision": PRECISION,
                "requests": stats["requests"],
            })
            return
        if path in ("/v1/models", "/models"):
            self._json(200, {"object": "list", "data": [{"id": "indextts-2.5", "object": "model"}, {"id": "tts-1", "object": "model"}]})
            return
        self._json(404, {"error": {"message": f"Not found: {path}"}})

    def do_POST(self):
        path = self.path.split("?", 1)[0].rstrip("/")
        if path not in ("/v1/audio/speech", "/audio/speech"):
            self._json(404, {"error": {"message": f"Not found: {path}"}})
            return
        if not self._check_auth():
            self._json(401, {"error": {"message": "Invalid Bearer token"}})
            return
        n = int(self.headers.get("Content-Length", "0") or "0")
        body = json.loads(self.rfile.read(n).decode("utf-8"))
        text = str(body.get("input") or "").strip()
        if not text:
            self._json(400, {"error": {"message": "Empty input"}})
            return
        lang = str(body.get("lang") or "").strip().upper() or detect_lang(text)
        req_fmt = str(body.get("response_format") or "wav").strip().lower()
        try:
            speed = max(0.6, min(1.6, float(body.get("speed", 1.0))))
        except Exception:
            speed = 1.0
        duration_factor = round(1.0 / speed, 3)
        t0 = time.time()
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tf:
            out_path = tf.name
        try:
            with infer_lock:
                tts.infer(
                    spk_audio_prompt=REF_WAV,
                    text=text,
                    output_path=out_path,
                    lang=lang,
                    duration_factor=duration_factor,
                )
                stats["requests"] += 1
            audio, content_type = transcode_audio(out_path, req_fmt)
        finally:
            if os.path.exists(out_path):
                os.remove(out_path)
        dt = time.time() - t0
        self.log_message("synth lang=%s fmt=%s chars=%d bytes=%d elapsed=%.2fs", lang, req_fmt, len(text), len(audio), dt)
        self.send_response(200)
        self._cors()
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(audio)))
        self.end_headers()
        self.wfile.write(audio)

srv = ThreadingHTTPServer((BIND_HOST, PORT), Handler)
print(f"[GPU-TTS] Ready on http://{BIND_HOST}:{PORT}/v1/audio/speech", flush=True)
srv.serve_forever()
'''

Path("/content/index-tts/colab_tts_worker_v2.py").write_text(server_code.strip() + "\n", encoding="utf-8")

env = os.environ.copy()
env["SHINOBU_TTS_API_KEY"] = API_KEY
env["SHINOBU_REF_WAV"] = REF_WAV
env["SHINOBU_TTS_PORT"] = str(PORT)
env["SHINOBU_TTS_BIND"] = "0.0.0.0" if TRANSPORT_MODE == "tailscale" else "127.0.0.1"
env["SHINOBU_GPU_PRECISION"] = PRECISION
env["SHINOBU_GPU_NAME"] = GPU_NAME

srv_proc = subprocess.Popen(
    ["uv", "run", "python", "colab_tts_worker_v2.py"],
    cwd="/content/index-tts",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in srv_proc.stdout:
    print(line, end="")
    if "/v1/audio/speech" in line and "Ready on http://" in line:
        break

transport_procs = []
public_base_url = None

if TRANSPORT_MODE == "tailscale":
    assert TAILSCALE_KEY, "Tailscale 模式需要配置 TS_AUTHKEY（建议放 Colab Secrets）！"
    ts_sock = "/tmp/tailscaled.sock"
    ts_state = "/tmp/tailscale.state"
    tsd_proc = subprocess.Popen(
        [
            "tailscaled",
            "--tun=userspace-networking",
            f"--state={ts_state}",
            f"--socket={ts_sock}",
            "--socks5-server=127.0.0.1:1055",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    transport_procs.append(tsd_proc)
    time.sleep(2.5)
    subprocess.run(
        [
            "tailscale",
            f"--socket={ts_sock}",
            "up",
            f"--authkey={TAILSCALE_KEY}",
            f"--hostname={TS_HOSTNAME}",
            "--accept-dns=false",
        ],
        check=True,
    )
    ts_ip = subprocess.check_output(
        ["tailscale", f"--socket={ts_sock}", "ip", "-4"],
        text=True,
    ).strip().splitlines()[0]
    # Userspace tailscaled serves incoming tailnet TCP via `tailscale serve`.
    serve_proc = subprocess.Popen(
        ["tailscale", f"--socket={ts_sock}", "serve", "--bg", f"http://127.0.0.1:{PORT}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    serve_out, _ = serve_proc.communicate(timeout=15)
    m_ts = re.search(r"https://[a-z0-9.-]+\.ts\.net", serve_out or "")
    public_base_url = m_ts.group(0) if m_ts else f"http://{ts_ip}:{PORT}"
    print(f"[Tailscale] Ready: {public_base_url} (ip={ts_ip})")
else:
    cf_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    transport_procs.append(cf_proc)
    t_start = time.time()
    for line in cf_proc.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            public_base_url = m.group(0)
            break
        if time.time() - t_start > 45:
            break
    assert public_base_url, "Cloudflare 隧道拉起超时，请重试本单元格。"

def _cloud_post(path: str, payload: dict) -> tuple[int, str]:
    url = f"{CLOUD_URL}{path}"
    headers = {"Content-Type": "application/json"}
    if CLOUD_TOKEN:
        headers["Authorization"] = f"Bearer {CLOUD_TOKEN}"
    req = urllib.request.Request(
        url,
        data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers=headers,
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            return int(resp.status), resp.read().decode("utf-8", errors="replace")
    except urllib.error.HTTPError as exc:
        return int(exc.code), exc.read(300).decode("utf-8", errors="replace")

def register_worker() -> bool:
    if not CLOUD_URL:
        return False
    payload = {
        "worker_id": WORKER_ID,
        "role": "colab_gpu",
        "base_url": public_base_url,
        "capabilities": ["tts"],
        "token": API_KEY,
        "ttl_sec": int(WORKER_TTL_SEC),
        "meta": {
            "gpu": GPU_NAME,
            "precision": PRECISION,
            "transport": TRANSPORT_MODE,
            "model": "indextts-2.5",
        },
    }
    code, text = _cloud_post("/api/workers/register", payload)
    print(f"[Registry] register -> HTTP {code} {text[:160]}", flush=True)
    return 200 <= code < 300

stop_hb = threading.Event()
started_at = time.time()

def _heartbeat_loop() -> None:
    while not stop_hb.wait(max(5, int(HEARTBEAT_INTERVAL_SEC))):
        if not CLOUD_URL:
            continue
        try:
            code, text = _cloud_post(
                "/api/workers/heartbeat",
                {
                    "worker_id": WORKER_ID,
                    "meta": {
                        "gpu": GPU_NAME,
                        "precision": PRECISION,
                        "uptime_sec": int(time.time() - started_at),
                    },
                },
            )
            if code == 404:
                print("[Registry] heartbeat 404 (cloud restarted?), re-registering...", flush=True)
                register_worker()
            elif not (200 <= code < 300):
                print(f"[Registry] heartbeat warn HTTP {code}: {text[:120]}", flush=True)
        except Exception as exc:
            print(f"[Registry] heartbeat error: {exc}", flush=True)

if CLOUD_URL:
    register_worker()
    hb_thread = threading.Thread(target=_heartbeat_loop, daemon=True)
    hb_thread.start()
else:
    print("[Registry] 未配置 CLOUD_BASE_URL，跳过自动报到（仅本地调试模式）。")

print("\n" + "=" * 70)
print(f"  SHINOBU COLAB TTS WORKER v2 LIVE! ({GPU_NAME} · {PRECISION})")
print(f"  Worker ID : {WORKER_ID}")
print(f"  Base URL  : {public_base_url}")
print(f"  API Key   : {API_KEY}")
print(f"  Cloud Reg : {CLOUD_URL or '(disabled)'}")
print("=" * 70 + "\n", flush=True)

try:
    for line in srv_proc.stdout:
        print(line, end="")
except KeyboardInterrupt:
    print("Stopping GPU TTS worker...")
finally:
    stop_hb.set()
    if CLOUD_URL:
        try:
            code, _ = _cloud_post("/api/workers/unregister", {"worker_id": WORKER_ID})
            print(f"[Registry] unregister -> HTTP {code}")
        except Exception as exc:
            print(f"[Registry] unregister skipped: {exc}")
    for proc in transport_procs:
        proc.terminate()
    srv_proc.terminate()
